# L23: 前端界面开发

> 使用 Streamlit 构建 Agent 交互界面

In [ ]:
# === 环境设置 ===
import sys
import os
from pathlib import Path

# 自动找到项目根目录
current_path = Path(os.getcwd())
project_path = None

for path in [current_path] + list(current_path.parents):
    if (path / "backend").exists():
        project_path = str(path)
        break

if project_path and project_path not in sys.path:
    sys.path.insert(0, project_path)

print(f"项目路径: {project_path}")
print(f"Python 版本: {sys.version.split()[0]}")

# 查看前端目录结构
frontend_dir = Path(project_path) / "frontend"
if frontend_dir.exists():
    print("\n=== 前端目录结构 ===\n")
    for item in sorted(frontend_dir.iterdir()):
        if item.is_file() and not item.name.startswith('.'):
            print(f"  {item.name}")

## 23.1 为什么选择 Streamlit？

| 特性 | 说明 |
|------|------|
| **纯 Python** | 不需要 HTML/CSS/JS |
| **快速开发** | 几行代码即可构建界面 |
| **自动刷新** | 代码改动立即反映 |
| **组件丰富** | 聊天、输入、显示等 |
| **适合原型** | Agent 项目快速演示 |
| **免费部署** | Streamlit Cloud 一键部署 |

## 23.2 Streamlit 基础组件

In [ ]:
# Streamlit 常用组件示例
import streamlit as st

# 组件列表
components = {
    "文本显示": [
        "st.title() - 标题",
        "st.header() - 二级标题",
        "st.text() - 普通文本",
        "st.markdown() - Markdown 格式",
        "st.write() - 万能输出",
    ],
    "输入组件": [
        "st.text_input() - 文本输入",
        "st.text_area() - 多行文本",
        "st.number_input() - 数字输入",
        "st.selectbox() - 下拉选择",
        "st.slider() - 滑块",
        "st.file_uploader() - 文件上传",
    ],
    "按钮": [
        "st.button() - 普通按钮",
        "st.download_button() - 下载按钮",
        "st.link_button() - 链接按钮",
    ],
    "布局": [
        "st.columns() - 多列布局",
        "st.tabs() - 选项卡",
        "st.sidebar() - 侧边栏",
        "st.container() - 容器",
    ],
    "状态": [
        "st.session_state - 会话状态",
        "st.spinner() - 加载提示",
        "st.progress() - 进度条",
        "st.error/info/success() - 消息框",
    ],
}

print("=== Streamlit 常用组件 ===\n")
for category, items in components.items():
    print(f"{category}:")
    for item in items:
        print(f"  - {item}")
    print()

## 23.3 聊天界面实现

In [ ]:
# 聊天界面示例代码
chat_app_code = '''
import streamlit as st
import requests

# 页面配置
st.set_page_config(
    page_title="Agent Chat",
    page_icon="🤖",
    layout="centered"
)

# 初始化会话状态
if "messages" not in st.session_state:
    st.session_state.messages = []

# 标题
st.title("🤖 Agent 对话助手")

# 侧边栏配置
with st.sidebar:
    st.header("⚙️ 设置")
    
    provider = st.selectbox(
        "LLM 提供商",
        ["deepseek", "openai", "anthropic"],
        index=0
    )
    
    temperature = st.slider(
        "Temperature",
        min_value=0.0,
        max_value=2.0,
        value=0.7,
        step=0.1
    )
    
    if st.button("清空对话"):
        st.session_state.messages = []

# 显示历史消息
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

# 用户输入
if prompt := st.chat_input("输入你的问题..."):
    # 显示用户消息
    with st.chat_message("user"):
        st.markdown(prompt)
    st.session_state.messages.append({"role": "user", "content": prompt})
    
    # 获取助手回复
    with st.chat_message("assistant"):
        with st.spinner("思考中..."):
            # 调用后端 API
            response = requests.post(
                "http://localhost:8000/api/chat",
                json={
                    "message": prompt,
                    "temperature": temperature
                }
            )
            
            if response.status_code == 200:
                assistant_message = response.json()["response"]
                st.markdown(assistant_message)
                st.session_state.messages.append({
                    "role": "assistant",
                    "content": assistant_message
                })
            else:
                st.error("请求失败，请检查后端服务")
'''

print("=== 聊天界面代码示例 ===\n")
print(chat_app_code)

## 23.4 查看项目前端实现

In [ ]:
# 查看项目前端代码
frontend_app_path = Path(project_path) / "frontend" / "app.py"

if frontend_app_path.exists():
    print("=== frontend/app.py 源码 ===\n")
    with open(frontend_app_path, encoding="utf-8") as f:
        content = f.read()
        print(content[:1500] + "\n... (源码已截断)")
else:
    print("app.py 文件不存在")

## 23.5 运行前端服务

In [ ]:
# 前端服务启动命令
print("=== 启动前端服务 ===\n")

print("基础运行:")
print("  streamlit run frontend/app.py")

print("\n使用 uv 运行:")
print("  uv run streamlit run frontend/app.py")

print("\n指定端口:")
print("  streamlit run frontend/app.py --server.port 8501")

print("\n开发模式（自动重载）:")
print("  streamlit run frontend/app.py --server.runOnSave true")

print("\n访问地址:")
print("  - 本地: http://localhost:8501")
print("  - 网络: http://<本机IP>:8501")

## 23.6 前端优化技巧

In [ ]:
# Streamlit 优化技巧
optimization_tips = {
    "性能优化": [
        "使用 @st.cache_data 缓存数据",
        "使用 @st.cache_resource 缓存资源",
        "避免在每次渲染时重复计算",
        "使用 st.session_state 存储状态",
    ],
    "用户体验": [
        "添加 st.progress 显示进度",
        "使用 st.spinner 加载提示",
        "合理使用 st.columns 布局",
        "添加 st.experimental_memo 缓存",
    ],
    "调试技巧": [
        "使用 st.json() 调试数据",
        "使用 st.write() 打印变量",
        "检查 st.session_state 内容",
        "使用 --logger.level=debug 运行",
    ],
}

print("=== Streamlit 优化技巧 ===\n")
for category, tips in optimization_tips.items():
    print(f"{category}:")
    for tip in tips:
        print(f"  - {tip}")
    print()

## 23.7 常见面试问题

**Q: Streamlit 和 React/Vue 有什么区别？**
- A:
  | 特性 | Streamlit | React/Vue |
  |------|-----------|-----------|
  | 开发语言 | Python | JavaScript/TS |
  | 开发速度 | 非常快 | 较慢 |
  | 定制性 | 有限 | 完全自由 |
  | 性能 | 一般 | 优秀 |
  | 适用场景 | 原型/内部工具 | 生产应用 |

**Q: 如何保持聊天历史？**
- A:
  1. 使用 `st.session_state` 存储消息列表
  2. 每次渲染时遍历显示历史消息
  3. 新消息追加到列表末尾
  4. 可选：持久化到文件/数据库

**Q: 如何处理流式输出？**
- A:
  1. 使用 `st.write_stream()` 显示流式内容
  2. 使用生成器逐步 yield 内容
  3. 配合后端的 SSE 接口
  4. 或者使用 `st.markdown()` 配合 placeholder 更新

---

## 总结

本课程学习了前端界面开发的核心知识：

| 知识点 | 说明 |
|--------|------|
| **Streamlit** | 纯 Python 的 Web 框架 |
| **基础组件** | 文本、输入、按钮、布局 |
| **聊天界面** | chat_message + chat_input |
| **会话状态** | session_state 管理历史 |
| **API 调用** | requests 库调用后端 |
| **优化技巧** | 缓存、进度、用户体验 |

**下一步**: L24 将学习完整项目部署！